<a href="https://colab.research.google.com/github/andrew11morozovtwo/assistant_bot1/blob/main/assistant_bot_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import telebot
import logging
from openai import OpenAI
import csv
from datetime import datetime
import os
import http.client
from urllib.parse import urlparse
from bs4 import BeautifulSoup
import io
import requests

# Настройка логирования для отладки и мониторинга работы бота
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# *** Важно: Замените 'YOUR_TELEGRAM_BOT_TOKEN' на токен вашего Telegram-бота ***
# Не храните токен в коде напрямую, используйте переменные окружения или config-файлы!
API_TOKEN = os.environ.get('TELEGRAM_BOT_TOKEN')  # Пример получения токена из переменной окружения

# Проверка наличия токена
if not API_TOKEN:
    logging.critical("Необходимо установить переменную окружения TELEGRAM_BOT_TOKEN с токеном вашего бота!")
    exit(1)

bot = telebot.TeleBot(API_TOKEN)

# *** Важно: Замените 'YOUR_OPENAI_API_KEY' на ваш API key от OpenAI ***
# Не храните API key в коде напрямую, используйте переменные окружения или config-файлы!
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')  # Пример получения API key из переменной окружения

# Проверка наличия API key
if not OPENAI_API_KEY:
    logging.critical("Необходимо установить переменную окружения OPENAI_API_KEY с вашим API key!")
    exit(1)

# Настройка OpenAI client
client = OpenAI(
    api_key=OPENAI_API_KEY,
    # base_url="https://api.proxyapi.ru/openai/v1", # Закомментировано, если не используется прокси
)

# Путь к файлу логов (CSV) на Google Диске или локально
file_path = '/content/gdrive/MyDrive/telegram_bot_logs.csv'  # Измените на свой путь при необходимости

# Словарь для хранения истории разговора с каждым пользователем
conversation_history = {}

# Функция для извлечения текста из URL
def extract_text_from_url(url):
    """
    Извлекает текст из веб-страницы по URL, используя библиотеки urllib и BeautifulSoup.
    """
    try:
        parsed_url = urlparse(url)

        if not parsed_url.netloc:
            return "Ошибка: Некорректный URL"

        conn = http.client.HTTPSConnection(parsed_url.netloc)
        path = parsed_url.path or "/"
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }

        conn.request("GET", path, headers=headers)
        response = conn.getresponse()

        if response.status == 200:
            page_content = response.read().decode("utf-8")
            soup = BeautifulSoup(page_content, 'html.parser')
            text_content = soup.get_text()
            cleaned_text = "\n".join(line.strip() for line in text_content.splitlines() if line.strip())
            conn.close()
            return cleaned_text
        else:
            conn.close()
            return f"Ошибка: {response.status}"

    except Exception as e:
        return f"Произошла ошибка: {e}"

# Обработчики сообщений Telegram
@bot.message_handler(commands=['start'])
def send_welcome(message):
    """
    Обработчик команды /start. Отправляет приветственное сообщение пользователю.
    """
    chat_id = message.chat.id
    # Инициализация истории разговора для нового чата
    if chat_id not in conversation_history:
        conversation_history[chat_id] = []
    bot.reply_to(message, "Добро пожаловать в канал 'Это не канал'! Как я могу помочь? Бот версии 21_01_2025 г")

@bot.message_handler(content_types=['text'])
def handle_text_message(message):
    """
    Обработчик текстовых сообщений. Извлекает текст из URL (если есть) и отправляет сообщение на обработку.
    """
    chat_id = message.chat.id
    user_message = message.text  # Текст сообщения пользователя
    message_type = 'text'  # Указываем тип сообщения как текстовое

    # Проверяем, содержит ли сообщение текст "http"
    if "http" in user_message:
        # Сохраняем исходный текст сообщения
        original_message = user_message

        # Извлекаем первую ссылку из текста
        import re
        url_match = re.search(r'(http[s]?://[^\s]+)', user_message)
        if url_match:
            url = url_match.group(0)  # Первая найденная ссылка

            # Пытаемся извлечь текст с веб-страницы
            extracted_text = extract_text_from_url(url)

            if extracted_text:
                # Объединяем текст сообщения с извлеченным текстом
                user_message = f"{original_message}\n\n{extracted_text}"
                process_message(message, user_message, message_type, chat_id)
            else:
                bot.reply_to(message, "Не удалось извлечь текст из ссылки.")
        else:
            bot.reply_to(message, "Ссылка не найдена в сообщении.")
    else:
        # Если сообщение не содержит "http", обрабатываем его как обычный текст
        process_message(message, user_message, message_type, chat_id)

@bot.message_handler(content_types=['photo'])
def handle_photo_message(message):
    """
    Обработчик сообщений с фотографиями. Получает описание изображения от OpenAI и отправляет сообщение на обработку.
    """
    chat_id = message.chat.id
    user_message = message.caption if message.caption else "Фото без подписи"
    message_type = 'photo'

    try:
        # Получаем file_id самой большой версии фотографии
        file_id = message.photo[-1].file_id

        # Получаем информацию о файле
        file_info = bot.get_file(file_id)
        file_path = file_info.file_path
        image_url = f'https://api.telegram.org/file/bot{API_TOKEN}/{file_path}'

        logging.info(f"URL изображения: {image_url}")  # Логируем URL

    except Exception as e:
        logging.error(f"Ошибка при получении URL изображения из Telegram: {e}")
        user_message += "\nНе удалось получить URL изображения."
        process_message(message, user_message, message_type, chat_id)
        return  # Выходим из функции, чтобы избежать дальнейших ошибок

    try:
        # Запрашиваем описание изображения у OpenAI
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Что на этом изображении? Дай краткое описание на русском языке."},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": image_url,
                            },
                        },
                    ],
                }
            ],
            max_tokens=300,
        )

        logging.info(f"Ответ от OpenAI Vision API: {response}")  # Логируем полный ответ

        # Извлекаем описание изображения из ответа OpenAI
        image_description = response.choices[0].message.content

        # Добавляем описание изображения к сообщению пользователя
        user_message += f"\nОписание изображения: {image_description}"

    except Exception as e:
        logging.error(f"Ошибка при обращении к OpenAI Vision API: {e}")
        user_message += "\nНе удалось получить описание изображения."

    process_message(message, user_message, message_type, chat_id)

@bot.message_handler(content_types=['video'])
def handle_video_message(message):
    """
    Обработчик сообщений с видео. Просто извлекает подпись (если есть) и отправляет сообщение на обработку.
    """
    chat_id = message.chat.id
    user_message = message.caption if message.caption else "Видео без подписи"
    message_type = 'video'
    process_message(message, user_message, message_type, chat_id)

# Обработка голосовых сообщений
@bot.message_handler(content_types=['voice'])
def handle_voice_message(message):
    """
    Обработчик голосовых сообщений. Транскрибирует аудио и объединяет с подписью (если есть).
    """
    chat_id = message.chat.id
    message_type = 'voice'

    try:
        # Получаем информацию о файле
        file_id = message.voice.file_id
        file_info = bot.get_file(file_id)
        file_path = file_info.file_path
        file_url = f'https://api.telegram.org/file/bot{API_TOKEN}/{file_path}'

        # Скачиваем файл
        response = requests.get(file_url)
        response.raise_for_status()  # Проверяем на ошибки

        # Открываем файл как бинарный
        audio_file = io.BytesIO(response.content)

        # Транскрибируем аудио
        transcription = client.audio.transcriptions.create(
            model="whisper-1",
            file=(file_path, audio_file)  # Передаем имя файла и объект BytesIO
        )

        # Получаем транскрибированный текст
        transcribed_text = transcription.text

        # Формируем сообщение пользователю: сначала подпись, потом транскрипция
        user_message = message.caption if message.caption else ""  # Получаем подпись
        user_message += f"\nТранскрипция аудио: {transcribed_text}"  # Добавляем транскрипцию

        # Отправляем текст пользователю
        #bot.reply_to(message, user_message)

        # Добавляем транскрипцию в историю разговора
        #process_message(message, user_message, message_type, chat_id)

    except Exception as e:
        logging.error(f"Ошибка при транскрибации аудио: {e}")
        bot.reply_to(message, "Произошла ошибка при транскрибации аудио.")
    process_message(message, user_message, message_type, chat_id)

# Обработка аудио сообщений
@bot.message_handler(content_types=['audio'])
def handle_audio_message(message):
    """
    Обработчик аудио сообщений (отличных от голосовых).  Транскрибирует аудио и объединяет с подписью (если есть).
    """
    chat_id = message.chat.id
    message_type = 'audio'

    try:
        # Получаем информацию о файле
        file_id = message.audio.file_id  # Изменено с message.voice.file_id на message.audio.file_id
        file_info = bot.get_file(file_id)
        file_path = file_info.file_path
        file_url = f'https://api.telegram.org/file/bot{API_TOKEN}/{file_path}'

        # Скачиваем файл
        response = requests.get(file_url)
        response.raise_for_status()  # Проверяем на ошибки

        # Открываем файл как бинарный
        audio_file = io.BytesIO(response.content)

        # Транскрибируем аудио
        transcription = client.audio.transcriptions.create(
            model="whisper-1",
            file=(file_path, audio_file)  # Передаем имя файла и объект BytesIO
        )

        # Получаем транскрибированный текст
        transcribed_text = transcription.text

         # Формируем сообщение пользователю: сначала подпись, потом транскрипция
        user_message = message.caption if message.caption else ""  # Получаем подпись
        user_message += f"\nТранскрипция аудио: {transcribed_text}"  # Добавляем транскрипцию

        # Отправляем текст пользователю
        #bot.reply_to(message, user_message)

        # Добавляем транскрипцию в историю разговора
        #process_message(message, user_message, message_type, chat_id)

    except Exception as e:
        logging.error(f"Ошибка при транскрибации аудио: {e}")
        bot.reply_to(message, "Произошла ошибка при транскрибации аудио.")
    process_message(message, user_message, message_type, chat_id)

@bot.message_handler(content_types=['poll'])
def handle_poll_message(message):
    """
    Обработчик сообщений с опросами. Просто извлекает вопрос опроса и отправляет сообщение на обработку.
    """
    chat_id = message.chat.id
    user_message = f"Опрос: {message.poll.question}"
    message_type = 'poll'
    process_message(message, user_message, message_type, chat_id)

# Функции для работы с логами
def initialize_log_file():
    """
    Создает файл логов, если он не существует, и записывает заголовки.
    """
    if not os.path.exists(file_path):
        with open(file_path, 'w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(['chat_id', 'datetime', 'message', 'message_type', 'ai_response'])

def log_to_file(chat_id, user_message, message_type, ai_response):
    """
    Записывает информацию о сообщении и ответе AI в файл логов.
    """
    current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')  # Текущее время
    with open(file_path, 'a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow([chat_id, current_time, user_message, message_type, ai_response])

# Основная функция обработки сообщений
def process_message(message, user_message, message_type, chat_id):
    """
    Основная функция обработки сообщений.  Отправляет запрос в OpenAI, получает ответ и отправляет его пользователю.
    Также ведет логи и сохраняет историю разговора.
    """
    # Проверяем, существует ли история для данного chat_id
    if chat_id not in conversation_history:
        conversation_history[chat_id] = []

    # Добавляем сообщение пользователя в историю разговора
    conversation_history[chat_id].append({"role": "user", "content": user_message})
    logging.info(f"Получено сообщение от пользователя: {user_message} (Тип: {message_type})")
    # Запрос к OpenAI с историей разговора
    try:
        chat_completion = client.chat.completions.create(
            model="gpt-3.5-turbo-1106",
            messages=conversation_history[chat_id] + [
                {"role": "system", "content": (
                    "Вы бот-администратор в телеграм-канале 'Это не канал'. Ваша задача — пересказывать на русском языке "
                    "подписчикам материалы, присылаемые в канал. Формируйте краткий (не более 3000 знаков) и интересный пересказ, "
                    "ориентируясь на следующие принципы:\n\n"
                    "1. Прочитайте пост. Если в посте указана ссылка, предположите, что она содержит дополнительную информацию. "
                    "Попробуйте дать пересказ, основываясь на теме, изложенной в посте, а также возможных контекстах.\n"
                    "2. Пересказ оформляйте структурно:\n"
                    "- Введение: кратко объясните, о чем материал и почему он важен.\n"
                    "- Основная часть: изложите ключевые моменты материала простым языком, подчеркивая суть. Разделяйте текст на абзацы.\n"
                    "- Заключение: сделайте выводы, предложите рекомендации или задайте вопрос для вовлечения подписчиков.\n"
                    "3. Если пост содержит только ссылку, составьте предположительный пересказ на основе общего контекста и доступной информации. "
                    "Укажите, что пересказ основан на интерпретации.\n"
                    "4. Указывайте источник информации в конце текста (например: 'Источник: ссылка из поста').\n\n"
                    "Общайтесь с читателями вежливо, от мужского лица, используя 'Вы'.\n\n"
                    "Включайте эмодзи для акцентирования ключевых моментов, таких как:\n"
                    "- 🔍 для выделения важных деталей,\n"
                    "- 📌 для ключевых тезисов,\n"
                    "- 🌟 для рекомендаций.\n\n"
                    "Следите за тем, чтобы текст был легко читаем на русском языке и не перегружен эмодзи. Старайтесь создавать увлекательные посты, чтобы подписчики захотели прочитать оригинал."
                )},
                {"role": "user", "content": user_message}
            ]
        )
        # Получаем ответ от AI
        ai_response = chat_completion.choices[0].message.content
        bot.reply_to(message, ai_response)

        # Логирование данных в файл
        log_to_file(chat_id, user_message, message_type, ai_response)

        # Добавляем ответ AI в историю разговора
        conversation_history[chat_id].append({"role": "assistant", "content": ai_response})
    except Exception as e:
        logging.error(f"Ошибка при обращении к OpenAI: {e}")
        bot.reply_to(message, "Извините, произошла ошибка при обработке вашего запроса.")

# Запуск бота
if __name__ == '__main__':
    initialize_log_file()  # Инициализация файла логов
    bot.polling(none_stop=True)
